In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Notebook_Spark") \
    .getOrCreate()

In [4]:
##Top 2 Products per Country
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("sales")

spark.sql("""
SELECT country, product, total_sales
FROM (
    SELECT 
        country,
        product,
        SUM(amount) AS total_sales,
        ROW_NUMBER() OVER (PARTITION BY country ORDER BY SUM(amount) DESC) AS rank
    FROM sales
    GROUP BY country, product
) t
WHERE rank <= 2
""").show()

+-------+-------+-----------+
|country|product|total_sales|
+-------+-------+-----------+
|Germany|      A|      12051|
|Germany|      C|      11879|
|  India|      C|       9172|
|  India|      A|       8596|
|     UK|      D|       8533|
|     UK|      B|       7836|
|    USA|      D|       9732|
|    USA|      C|       9004|
+-------+-------+-----------+



In [5]:
###Top 3 Products by Total Sales
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("sales")
spark.sql("""
SELECT product, SUM(amount) AS total_sales
FROM sales
GROUP BY product
ORDER BY total_sales DESC
LIMIT 3
""").show()


+-------+-----------+
|product|total_sales|
+-------+-----------+
|      D|      36707|
|      A|      36605|
|      C|      36016|
+-------+-----------+



In [6]:
df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("sales")
spark.sql("""
SELECT country, amount AS second_highest_sale
FROM (
    SELECT 
        country,
        amount,
        ROW_NUMBER() OVER (PARTITION BY country ORDER BY amount DESC) AS rank
    FROM sales
) t
WHERE rank = 2
""").show()

+-------+-------------------+
|country|second_highest_sale|
+-------+-------------------+
|Germany|                495|
|  India|                492|
|     UK|                487|
|    USA|                490|
+-------+-------------------+



In [7]:
df = spark.read.text("app_logs.txt")
df.createOrReplaceTempView("raw_logs")

In [8]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW logs AS
SELECT
    get(parts, 0) AS date,
    get(parts, 1) AS time,
    get(parts, 2) AS level,
    get(parts, 3) AS user,
    get(parts, 4) AS action,
    get(parts, 5) AS status_or_item
FROM (
    SELECT split(value, ' ') AS parts
    FROM raw_logs
) t
""")

DataFrame[]

In [9]:
#Count of Each Log Level
spark.sql("""
SELECT level, COUNT(*) AS count
FROM logs
GROUP BY level
""").show()

+-----+-----+
|level|count|
+-----+-----+
| INFO|   12|
|ERROR|    5|
| WARN|    3|
+-----+-----+



In [10]:
#Most Active User
spark.sql("""
SELECT user, COUNT(*) AS total_actions
FROM logs
GROUP BY user
ORDER BY total_actions DESC
LIMIT 1
""").show()

+-----+-------------+
| user|total_actions|
+-----+-------------+
|user1|            4|
+-----+-------------+



In [11]:
#User with Maximum Errors
spark.sql("""
SELECT user, COUNT(*) AS error_count
FROM logs
WHERE level = 'ERROR'
GROUP BY user
ORDER BY error_count DESC
LIMIT 1
""").show()

+-----+-----------+
| user|error_count|
+-----+-----------+
|user2|          2|
+-----+-----------+



In [12]:
# users with Failed Login Attempts
spark.sql("""
SELECT COUNT(*) AS failed_logins
FROM logs
WHERE action = 'login'
AND status_or_item = 'failed'
""").show()

+-------------+
|failed_logins|
+-------------+
|            2|
+-------------+



In [13]:
#Most Purchased Item
spark.sql("""
SELECT status_or_item AS item,
       COUNT(*) AS purchase_count
FROM logs
WHERE action = 'purchase'
GROUP BY status_or_item
ORDER BY purchase_count DESC
LIMIT 1
""").show()

+-------+--------------+
|   item|purchase_count|
+-------+--------------+
|item555|             1|
+-------+--------------+



In [14]:
#Users with Both Login Success & Failure
spark.sql("""
SELECT DISTINCT l1.user
FROM logs l1
JOIN logs l2
ON l1.user = l2.user

WHERE l1.action = 'login'
AND l1.status_or_item = 'success'

AND l2.action = 'login'
AND l2.status_or_item = 'failed'
""").show()


+----+
|user|
+----+
+----+



In [15]:
#Consecutive Failures
spark.sql("""
SELECT DISTINCT user

FROM (

    SELECT
        user,
        action,
        status_or_item,

        LAG(status_or_item)
        OVER (
            PARTITION BY user
            ORDER BY time
        ) AS prev_status

    FROM logs

) t

WHERE status_or_item = 'failed'
AND prev_status = 'failed'
""").show()

+-----+
| user|
+-----+
|user2|
+-----+



In [26]:
#Activity
# Top 10 revenue-generating users
df = spark.read.csv("customers_dataset.csv", header=True, inferSchema=True)
df1 = spark.read.csv("orders_dataset.csv", header=True, inferSchema=True)
df2 = spark.read.csv("products_dataset.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("customers")
df1.createOrReplaceTempView("orders")
df2.createOrReplaceTempView("products")

spark.sql("""
SELECT c.customer_name, c.customer_id, SUM(p.price * o.quantity) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_revenue DESC
LIMIT 10
""").show()

+----------------+-----------+-------------+
|   customer_name|customer_id|total_revenue|
+----------------+-----------+-------------+
|Theresa Ferguson|          6|      1151252|
|Shirley Harrison|         93|      1107491|
|    Shane Miller|        106|       766171|
|   Darrell Jones|         11|       751203|
|Kristine Schmidt|         48|       713509|
|     David Brown|        107|       701956|
|      Karla Diaz|         24|       694154|
|   Alan Anderson|        114|       687995|
|   Karen Pittman|         79|       663500|
|  Nicole Johnson|         54|       641409|
+----------------+-----------+-------------+



In [52]:
#Monthly Revenue Trend - Find month-wise revenue trend. (use DATE_FORMAT function to choose YYYY-MM)
spark.sql("""
SELECT DATE_FORMAT(order_date, 'yyyy-MM') AS month, SUM(sales_amount) AS monthly_revenue
FROM orders o
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
ORDER BY month
""").show()

+-------+------------------+
|  month|   monthly_revenue|
+-------+------------------+
|2025-05|2780923.4999999986|
|2025-06|3047238.7300000004|
|2025-07|2519717.0100000002|
|2025-08|3116055.6300000004|
|2025-09|2073947.1400000001|
|2025-10|2432460.0199999996|
|2025-11|3146286.3499999987|
|2025-12|        2698370.84|
|2026-01|2728039.2299999995|
|2026-02|        1875279.53|
|2026-03| 3167027.819999999|
|2026-04|2734291.4299999997|
|2026-05|         738030.75|
+-------+------------------+



In [33]:
#cummulative revenue over time

spark.sql("""
SELECT order_date, SUM(sales_amount) OVER (ORDER BY order_date) AS cumulative_revenue
FROM orders o
""").show()

+----------+------------------+
|order_date|cumulative_revenue|
+----------+------------------+
|2025-05-07|179179.03999999998|
|2025-05-07|179179.03999999998|
|2025-05-07|179179.03999999998|
|2025-05-07|179179.03999999998|
|2025-05-08|260538.41999999998|
|2025-05-08|260538.41999999998|
|2025-05-09|         336093.66|
|2025-05-10| 826898.4099999999|
|2025-05-10| 826898.4099999999|
|2025-05-10| 826898.4099999999|
|2025-05-10| 826898.4099999999|
|2025-05-10| 826898.4099999999|
|2025-05-12|         1156169.6|
|2025-05-12|         1156169.6|
|2025-05-12|         1156169.6|
|2025-05-12|         1156169.6|
|2025-05-14|        1226728.55|
|2025-05-16|1237346.1700000002|
|2025-05-17|1297303.1000000003|
|2025-05-17|1297303.1000000003|
+----------+------------------+
only showing top 20 rows


In [36]:
#Previous Order Analysis - Compare customer current purchase with previous purchase.(use LAG )

spark.sql("""
SELECT customer_id,
          order_id,
          order_date,
          sales_amount AS current_purchase_amount,
          LAG(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_purchase_amount
FROM orders o
""").show()

+-----------+--------+----------+-----------------------+------------------------+
|customer_id|order_id|order_date|current_purchase_amount|previous_purchase_amount|
+-----------+--------+----------+-----------------------+------------------------+
|          1|     251|2025-09-19|               126064.4|                    NULL|
|          1|     424|2025-10-08|               30719.96|                126064.4|
|          2|     443|2025-10-06|              108951.48|                    NULL|
|          2|     306|2025-11-21|               31602.42|               108951.48|
|          2|     174|2025-12-04|     196283.40000000002|                31602.42|
|          2|     378|2026-03-01|                59844.0|      196283.40000000002|
|          3|     402|2025-06-03|               32065.68|                    NULL|
|          3|     347|2025-09-18|     2446.9900000000002|                32065.68|
|          3|     400|2026-01-13|     22083.600000000002|      2446.9900000000002|
|   

In [37]:
#Next Purchase Prediction using LEAD - Predict next customer purchase amount.

spark.sql("""
SELECT customer_id,
          order_id,
          order_date,
          sales_amount AS current_purchase_amount,
          LEAD(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS next_purchase_amount
FROM orders o
""").show()

+-----------+--------+----------+-----------------------+--------------------+
|customer_id|order_id|order_date|current_purchase_amount|next_purchase_amount|
+-----------+--------+----------+-----------------------+--------------------+
|          1|     251|2025-09-19|               126064.4|            30719.96|
|          1|     424|2025-10-08|               30719.96|                NULL|
|          2|     443|2025-10-06|              108951.48|            31602.42|
|          2|     306|2025-11-21|               31602.42|  196283.40000000002|
|          2|     174|2025-12-04|     196283.40000000002|             59844.0|
|          2|     378|2026-03-01|                59844.0|                NULL|
|          3|     402|2025-06-03|               32065.68|  2446.9900000000002|
|          3|     347|2025-09-18|     2446.9900000000002|  22083.600000000002|
|          3|     400|2026-01-13|     22083.600000000002|           131495.24|
|          3|     101|2026-03-12|              13149

In [38]:
#Customer Retention Analysis - Find customers who ordered in consecutive months.

spark.sql("""
SELECT DISTINCT customer_id
FROM (
    SELECT customer_id,
           order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
) t
WHERE DATEDIFF(order_date, previous_order_date) = 30
""").show()

+-----------+
|customer_id|
+-----------+
|         52|
|         56|
+-----------+



In [43]:
#Highest Selling Product Per Category - Find top-selling product in each category

spark.sql("""
SELECT category,
    product_name AS product,
    total_sales
FROM (
    SELECT p.category AS category,
        p.product_name AS product_name,
        SUM(o.sales_amount) AS total_sales,
        ROW_NUMBER() OVER (PARTITION BY p.category ORDER BY SUM(o.sales_amount) DESC) AS rank
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, p.product_name
) t
WHERE rank = 1
""").show()

+-----------+--------------+-----------------+
|   category|       product|      total_sales|
+-----------+--------------+-----------------+
|Electronics|Worker Product|742170.8500000001|
|    Fashion| Build Product|        1215711.4|
|  Furniture| Place Product|       1193462.47|
|    Grocery|  Fill Product|763266.6300000001|
|     Sports| Visit Product|749711.3200000001|
+-----------+--------------+-----------------+



In [45]:
#Average Shipping Delay - Logistics team wants average delivery delay.( find AVG and DATEDIFF ).

spark.sql("""
SELECT AVG(DATEDIFF(ship_date, order_date)) AS average_shipping_delay
FROM orders o
""").show()

+----------------------+
|average_shipping_delay|
+----------------------+
|                 5.672|
+----------------------+



In [46]:
#Customers With No Orders - Marketing team wants inactive customers.

spark.sql("""
SELECT c.customer_id, c.customer_name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
""").show()

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|         37|   Robin Cobb|
+-----------+-------------+



In [47]:
#Rank Customers Based on Revenue- Basically rank customer who generates high revenue.

spark.sql("""
SELECT customer_id, customer_name, total_revenue,
RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM (
    SELECT c.customer_id, c.customer_name, SUM(p.price * o.quantity) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
    GROUP BY c.customer_id, c.customer_name
) t
""").show()

+-----------+--------------------+-------------+------------+
|customer_id|       customer_name|total_revenue|revenue_rank|
+-----------+--------------------+-------------+------------+
|          6|    Theresa Ferguson|      1151252|           1|
|         93|    Shirley Harrison|      1107491|           2|
|        106|        Shane Miller|       766171|           3|
|         11|       Darrell Jones|       751203|           4|
|         48|    Kristine Schmidt|       713509|           5|
|        107|         David Brown|       701956|           6|
|         24|          Karla Diaz|       694154|           7|
|        114|       Alan Anderson|       687995|           8|
|         79|       Karen Pittman|       663500|           9|
|         54|      Nicole Johnson|       641409|          10|
|        118|        John Johnson|       633376|          11|
|         85|       Courtney Bell|       628405|          12|
|         45|        Jason Nelson|       608911|          13|
|       

In [63]:
#Detect Revenue Drop - Find customers whose sales reduced compared to previous purchase.(use LAG and later filter curr sales < prev sales), display the customer name and customer id as well., keep the customer only once

spark.sql("""
SELECT DISTINCT customer_id, customer_name
FROM (
    SELECT c.customer_id, c.customer_name, o.order_date, o.sales_amount,
           LAG(o.sales_amount) OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS previous_sales
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
) t
WHERE sales_amount < previous_sales
""").show()

+-----------+----------------+
|customer_id|   customer_name|
+-----------+----------------+
|          1| Zachary Randall|
|          2|   Morgan Wilson|
|          3|      Troy Brown|
|          6|Theresa Ferguson|
|          7|  Yvonne Carroll|
|          8|   Michael Sloan|
|          9|   Mark Thompson|
|         10|     Tara Walker|
|         11|   Darrell Jones|
|         12| Anthony Delgado|
|         13|  William Bailey|
|         14|  Lisa Lopez DDS|
|         15|  Emily Galloway|
|         16|      Ryan Cohen|
|         18|   Jaime Vasquez|
|         19|   Hector Gordon|
|         20|    Paul Stevens|
|         21|    Jordan Jones|
|         22| Aaron Mcconnell|
|         24|      Karla Diaz|
+-----------+----------------+
only showing top 20 rows


In [64]:
# Day-wise Revenue Analysis - Find which weekday generates highest revenue.(use DAYNAME to get the name)

spark.sql("""
SELECT DAYNAME(order_date) AS weekday, SUM(sales_amount) AS total_revenue
FROM orders o
GROUP BY DAYNAME(order_date)
ORDER BY total_revenue DESC
LIMIT 1
""").show()

+-------+-----------------+
|weekday|    total_revenue|
+-------+-----------------+
|    Mon|5655873.800000001|
+-------+-----------------+



In [81]:
# Moving average sales
spark.sql("""
SELECT order_date,
       daily_sales,
       ROUND(
           AVG(daily_sales) OVER (
               ORDER BY CAST(order_date AS DATE)
               RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
           ),
           2
       ) AS moving_average
FROM (
    SELECT CAST(order_date AS DATE) AS order_date,
           SUM(sales_amount) AS daily_sales
    FROM orders
    GROUP BY CAST(order_date AS DATE)
) t
ORDER BY order_date
""").show()

+----------+------------------+--------------+
|order_date|       daily_sales|moving_average|
+----------+------------------+--------------+
|2025-05-07|179179.03999999998|     179179.04|
|2025-05-08|          81359.38|     130269.21|
|2025-05-09| 75555.23999999999|     112031.22|
|2025-05-10|         490804.75|      206724.6|
|2025-05-12|329271.19000000006|     231233.92|
|2025-05-14|          70558.95|      209509.9|
|2025-05-16|10617.619999999999|     225313.13|
|2025-05-17| 59956.92999999999|     117601.17|
|2025-05-19|         254054.68|      98797.05|
|2025-05-21| 76966.06000000001|     100398.82|
|2025-05-22|         162546.86|     112828.43|
|2025-05-23|          54300.09|     121564.92|
|2025-05-24|465520.32999999996|      202677.6|
|2025-05-25|262350.39999999997|     212623.07|
|2025-05-26|          114203.0|     189314.46|
|2025-05-27|          15765.52|     164521.75|
|2025-05-30|          49703.46|     181508.54|
|2025-05-31|           28210.0|      94046.48|
|2025-06-01| 

In [85]:
#Revenue Contribution % - Find each customer's contribution percentage.( ROUND 2)

spark.sql("""
SELECT 
    customer_id,
    SUM(sales_amount) AS total_revenue,
    ROUND(
        (SUM(sales_amount) * 100.0) / (SELECT SUM(sales_amount) FROM orders),
        2
    ) AS revenue_contribution_percentage
FROM orders
GROUP BY customer_id
""").show()

+-----------+------------------+-------------------------------+
|customer_id|     total_revenue|revenue_contribution_percentage|
+-----------+------------------+-------------------------------+
|         31|          182534.2|                           0.55|
|         85|485519.66000000003|                           1.47|
|         65|         417792.07|                           1.26|
|         53|         222359.14|                           0.67|
|         78| 494396.5199999999|                            1.5|
|        108|         182294.28|                           0.55|
|         34|         221218.12|                           0.67|
|        101|         345966.48|                           1.05|
|        115|         145863.62|                           0.44|
|         81|         117330.44|                           0.35|
|         28|          257819.9|                           0.78|
|         76|         314220.92|                           0.95|
|         26|          39

In [ ]:
# Fraud Detection:
# Fraud ONLY if ALL conditions are true:
# 1. Same customer
# 2. Multiple orders
# 3. Orders placed within 5 minutes
# 4. High amount (> threshold)

from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp, count

# Define threshold
high_amount_threshold = 10000

# Window by customer ordered by transaction time
window_spec = Window.partitionBy("customer_id").orderBy("order_timestamp")

fraud_orders = orders \
    .withColumn("prev_order_time", lag("order_timestamp").over(window_spec)) \
    .withColumn(
        "time_diff_minutes",
        (unix_timestamp("order_timestamp") - unix_timestamp("prev_order_time")) / 60
    ) \
    .withColumn(
        "order_count",
        count("*").over(Window.partitionBy("customer_id"))
    ) \
    .filter(
        (col("order_count") > 1) &                 # Multiple orders
        (col("time_diff_minutes") <= 5) &          # Within 5 minutes
        (col("amount") > high_amount_threshold)    # High amount
    )

fraud_orders.show(truncate=False)

In [91]:
#Customer Churn Prediction Logic - Find customers inactive for last 90 days.

spark.sql("""
SELECT t.customer_id, t.customer_name
FROM (
    SELECT 
        c.customer_id,
        c.customer_name,
        MAX(o.order_date) AS last_order_date
    FROM customers c
    LEFT JOIN orders o 
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
) t
WHERE DATEDIFF(CURRENT_DATE(), last_order_date) >= 90
""").show()

+-----------+-------------------+
|customer_id|      customer_name|
+-----------+-------------------+
|         59|         Jim Deleon|
|        112|       John Serrano|
|         63|    Michael Bridges|
|         20|       Paul Stevens|
|        115|     Marissa Watson|
|        119|      Joseph Watson|
|          1|    Zachary Randall|
|         32|     Robert Jackson|
|         47|         John Rivas|
|         60|      Debra Ramirez|
|         18|      Jaime Vasquez|
|         75|         Michael Li|
|         99|   Cassandra Conner|
|          7|     Yvonne Carroll|
|         77|Clifford Williamson|
|          4|     Crystal Miller|
|         66|         Anna Walsh|
|          5|      Marisa Miller|
|         42|    Valerie Johnson|
|         35|      Ashley Little|
+-----------+-------------------+
only showing top 20 rows


In [102]:
# Products bought together - Find products that are frequently bought together. check customer_id with different productid on the same day

spark.sql("""
SELECT 
    o1.product_id AS product_id_1,
    o2.product_id AS product_id_2,
    COUNT(*) AS times_bought_together
FROM orders o1
JOIN orders o2
ON o1.customer_id = o2.customer_id
AND o1.order_date = o2.order_date
AND o1.product_id < o2.product_id
GROUP BY o1.product_id, o2.product_id
ORDER BY times_bought_together DESC
LIMIT 10
""").show()

+------------+------------+---------------------+
|product_id_1|product_id_2|times_bought_together|
+------------+------------+---------------------+
|          22|          33|                    1|
|           3|          10|                    1|
|           4|           6|                    1|
+------------+------------+---------------------+



In [105]:
#Inventory Risk Detection - Find products that may go out of stock soon. logic is check the quantity of the product being sold in the orders and check the stock quantity of the product in products table, if the stock quantity is less than 10 then it is at risk of going out of stock.

spark.sql("""
SELECT p.product_id, p.product_name, p.stock_quantity,
       SUM(o.quantity) AS total_sold
FROM products p
JOIN orders o ON p.product_id = o.product_id
GROUP BY p.product_id, p.product_name, p.stock_quantity
HAVING p.stock_quantity < 50
ORDER BY total_sold DESC
""").show()

+----------+---------------+--------------+----------+
|product_id|   product_name|stock_quantity|total_sold|
+----------+---------------+--------------+----------+
|        55|    Bad Product|            30|        46|
|        51|Brother Product|            46|        35|
|        48|Million Product|            46|        27|
|        36|Section Product|            37|        26|
|        57|   Wear Product|            20|        19|
|        30|  Throw Product|            37|        17|
|        45|Message Product|            11|         8|
+----------+---------------+--------------+----------+



In [106]:
#Second Highest Revenue Customer Per State

spark.sql("""
SELECT
    customer_id,
    customer_name,
    total_revenue,
    revenue_rank
FROM (
    SELECT 
        customer_id,
        customer_name,
        state,
        total_revenue,
        RANK() OVER (PARTITION BY state ORDER BY total_revenue DESC) AS revenue_rank
    FROM (
        SELECT c.customer_id, c.customer_name, c.state, SUM(p.price * o.quantity) AS total_revenue
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN products p ON o.product_id = p.product_id
        GROUP BY c.customer_id, c.customer_name, c.state
    ) t1
) t2
WHERE revenue_rank = 2
""").show()

+-----------+----------------+-------------+------------+
|customer_id|   customer_name|total_revenue|revenue_rank|
+-----------+----------------+-------------+------------+
|        107|     David Brown|       701956|           2|
|         93|Shirley Harrison|      1107491|           2|
|         54|  Nicole Johnson|       641409|           2|
|        114|   Alan Anderson|       687995|           2|
|         38|     Sarah Klein|       542238|           2|
+-----------+----------------+-------------+------------+



In [113]:
#find revenue spike detection logic is check the revenue of two consecutive days, if the revenue is greater than the average of previous day then it is a revenue spike., also if greater than null

spark.sql("""
SELECT order_date, daily_revenue,
       LAG(daily_revenue) OVER (ORDER BY order_date) AS previous_day_revenue,
       CASE 
           WHEN previous_day_revenue IS NULL THEN 'Spike'  -- First day, no previous revenue
           WHEN daily_revenue > previous_day_revenue THEN 'Spike'
           ELSE 'No Spike'
       END AS revenue_spike
FROM (
    SELECT CAST(order_date AS DATE) AS order_date,
           SUM(sales_amount) AS daily_revenue
    FROM orders
    GROUP BY CAST(order_date AS DATE)
) t
ORDER BY order_date
""").show()

+----------+------------------+--------------------+-------------+
|order_date|     daily_revenue|previous_day_revenue|revenue_spike|
+----------+------------------+--------------------+-------------+
|2025-05-07|179179.03999999998|                NULL|        Spike|
|2025-05-08|          81359.38|  179179.03999999998|     No Spike|
|2025-05-09| 75555.23999999999|            81359.38|     No Spike|
|2025-05-10|         490804.75|   75555.23999999999|        Spike|
|2025-05-12|329271.19000000006|           490804.75|     No Spike|
|2025-05-14|          70558.95|  329271.19000000006|     No Spike|
|2025-05-16|10617.619999999999|            70558.95|     No Spike|
|2025-05-17| 59956.92999999999|  10617.619999999999|        Spike|
|2025-05-19|         254054.68|   59956.92999999999|        Spike|
|2025-05-21| 76966.06000000001|           254054.68|     No Spike|
|2025-05-22|         162546.86|   76966.06000000001|        Spike|
|2025-05-23|          54300.09|           162546.86|     No Sp